# LMA Phase 3: BHOJPURI Reasoning Finetuning

Finetunes the Phase 2 pretrained checkpoint on the synthetic comparative-reasoning QA set, using the real finetuning code imported from the same bundle used for pretraining (`kspsvln/lma-bhojpuri-phase2`).


In [1]:
# ===== CONFIG-ONLY CELL: Modify these before running =====
ROOT_DIR = "/kaggle/input/datasets/kspsvln/lma-bhojpuri-phase2"  # Same code+config bundle as pretraining (dataset-metadata.json id: kspsvln/lma-bhojpuri-phase2)
PRETRAINED_CKPT = "/kaggle/input/datasets/kspsvln/checkpoint-bhojpuri/checkpoints/checkpoint_best.pt"  # Phase 2 pretrained checkpoint -- attach the checkpoint-bhojpuri dataset as an input
OUT_DIR = "/kaggle/working/finetune_checkpoints"   # Save finetuned checkpoints here
CHECK_DIR = None                                    # Resume finetuning from a previous finetune session (if available)

LANGUAGE = "bhojpuri"

# Hyperparameters: None means use config JSON defaults (configs/finetune_config.json)
# Must match finetune.py's actual argparse/run_finetuning fields.
hp = dict(
    batch_size=None,
    learning_rate=None,
    num_epochs=None,
    warmup_steps=None,
    weight_decay=None,
    amp=True,
)


In [2]:
import sys
import os
import argparse
from pathlib import Path

# Fail fast if ROOT_DIR isn't actually where the bundle is mounted -- sys.path.insert()
# silently accepts a bad path, so a wrong ROOT_DIR would otherwise only surface later as a
# confusing "ModuleNotFoundError: No module named 'finetune'" at the import cell.
root_path = Path(ROOT_DIR)
expected_entry = root_path / "finetune" / "finetune.py"
if not expected_entry.exists():
    kaggle_input = Path("/kaggle/input")
    available = sorted(p.name for p in kaggle_input.iterdir()) if kaggle_input.exists() else []
    listing = "\n".join(f"  {p}" for p in sorted(root_path.iterdir())) if root_path.exists() else "  (ROOT_DIR does not exist)"
    raise RuntimeError(
        f"Expected {expected_entry} but it's not there.\n"
        f"ROOT_DIR = {ROOT_DIR}\n"
        f"Contents of ROOT_DIR:\n{listing}\n"
        f"Datasets attached under /kaggle/input/: {available}\n"
        "Check that the lma-bhojpuri-phase2 bundle (with the finetune/ code and data already "
        "in it) is attached as an input to this notebook (Add Input) and that ROOT_DIR above "
        "matches its actual mounted path -- it may be nested one level deeper depending on "
        "how it was uploaded."
    )

if not Path(PRETRAINED_CKPT).exists():
    raise RuntimeError(
        f"PRETRAINED_CKPT={PRETRAINED_CKPT} does not exist. Attach the checkpoint-bhojpuri "
        "dataset (kspsvln/checkpoint-bhojpuri) as an input to this notebook, or update "
        "PRETRAINED_CKPT above to wherever your pretrained checkpoint is actually mounted."
    )

# Setup path to import from bundle
sys.path.insert(0, ROOT_DIR)
os.chdir('/kaggle/working')  # For checkpoint/log output

# Install tokenizers if needed
import subprocess
subprocess.run(['pip', 'install', 'tokenizers'], capture_output=True)

print(f'Root: {ROOT_DIR}')
print(f'Pretrained checkpoint: {PRETRAINED_CKPT}')
print(f'Output: {OUT_DIR}')
print(f'Resume: {CHECK_DIR}')


Root: /kaggle/input/datasets/kspsvln/lma-bhojpuri-phase2
Pretrained checkpoint: /kaggle/input/datasets/kspsvln/checkpoint-bhojpuri/checkpoints/checkpoint_best.pt
Output: /kaggle/working/finetune_checkpoints
Resume: None


In [3]:
# Import the real finetuning code (no reimplementation)
from finetune.finetune import run_finetuning

print('✅ Imported finetuning code from bundle')


✅ Imported finetuning code from bundle


In [4]:
# Build command-line arguments by mimicking finetune.py's argparse
# Filter out None hyperparams (use config JSON defaults)
args_dict = {k: v for k, v in hp.items() if v is not None}

# Handle resume: compute resume_from path (resuming a FINETUNE session, separate from PRETRAINED_CKPT)
resume_from = None
if CHECK_DIR and Path(CHECK_DIR).exists():
    potential_ckpt = Path(CHECK_DIR) / 'checkpoint_last.pt'
    if potential_ckpt.exists():
        resume_from = str(potential_ckpt)
        print(f'Will resume finetuning from: {resume_from}')

# Create args namespace (fields must match finetune.py's run_finetuning() override_config exactly)
args = argparse.Namespace(
    batch_size=args_dict.get('batch_size', None),
    learning_rate=args_dict.get('learning_rate', None),
    num_epochs=args_dict.get('num_epochs', None),
    warmup_steps=args_dict.get('warmup_steps', None),
    weight_decay=args_dict.get('weight_decay', None),
    amp=args_dict.get('amp', True),
    pretrained_ckpt=PRETRAINED_CKPT,
    resume_from=resume_from,
)

print('✅ Arguments prepared')


✅ Arguments prepared


In [5]:
# Run finetuning with the real Trainer subclass (checkpointing, logging, AMP, masked QA loss, etc. all included)
# data_dir defaults to ROOT_DIR/finetune/data -- the QA dataset is bundled directly with the code
# since it's small (~10K short examples), no separate data input needed unlike pretraining's raw corpus.
print('\n' + '='*60)
print(f'Starting {LANGUAGE.upper()} reasoning finetuning...')
print('='*60 + '\n')

run_finetuning(args, root_dir=ROOT_DIR, data_dir=None, output_dir=OUT_DIR)

print('\n' + '='*60)
print('✅ Finetuning complete!')
print('='*60)



Starting BHOJPURI reasoning finetuning...

Device: cuda

Finetuning config:
  language: bhojpuri
  training_phase: reasoning_finetuning
  batch_size: 4
  learning_rate: 2e-05
  weight_decay: 0.01
  num_epochs: 20
  warmup_steps: 50
  optimizer: adamw
  scheduler: cosine_with_warmup
  loss_function: cross_entropy
  max_grad_norm: 1.0
  seed: 42
  device: auto
  num_workers: 2
  amp: True
  description: Phase 3 Bhojpuri reasoning finetuning: starts from the Phase 2 pretrained checkpoint, tokenizer/vocab fixed, finetunes on the synthetic comparative-reasoning QA set (8000 train examples, see finetune/data/). Loss is masked to answer tokens only. Lower learning_rate than pretraining since this is adapting an already-trained model to a narrow templated task, not learning language from scratch. num_epochs raised to 20 (full pass over all 8000 train examples per epoch) since val loss on the prior 5-epoch run bottomed out around epoch 1-2 then rose (overfitting on the templated data) -- finet

Training:  50%|████▉     | 1999/4000 [01:58<02:07, 15.64it/s, loss=4.8] 

  Saving checkpoint to /kaggle/working/finetune_checkpoints/checkpoint_last.pt


Training: 100%|█████████▉| 3999/4000 [03:56<00:00, 16.96it/s, loss=4.58]

  Saving checkpoint to /kaggle/working/finetune_checkpoints/checkpoint_last.pt


...

Training:  50%|████▉     | 1999/4000 [02:00<01:52, 17.77it/s, loss=0.48] 

  Saving checkpoint to /kaggle/working/finetune_checkpoints/checkpoint_last.pt


Training: 100%|█████████▉| 3999/4000 [04:00<00:00, 14.17it/s, loss=0.00189]

  Saving checkpoint to /kaggle/working/finetune_checkpoints/checkpoint_last.pt


Epoch 20: train_loss=0.3835 val_loss=9.6992 val_ppl=16304.51
  Saving checkpoint to /kaggle/working/finetune_checkpoints/checkpoint_last.pt

✓ Finetuning complete!
  Best val_loss: 8.3672
  Best val_ppl: 4303.41
✓ Loaded checkpoint_best.pt (step 4000) for final test evaluation

✓ Finetuned test exact-match accuracy (best checkpoint): 0.1640
✓ Saved summary to /kaggle/working/finetune_checkpoints/finetune_summary.json

✅ Finetuning complete!


In [6]:
# Final summary
import glob
import json

checkpoints = sorted(glob.glob(f'{OUT_DIR}/checkpoint_*.pt'))
logs = sorted(glob.glob(f'{OUT_DIR}/*.log'))
summary_path = Path(OUT_DIR) / 'finetune_summary.json'

print(f'\n📊 Finetuning outputs:')
if checkpoints:
    print(f'  Checkpoints:')
    for ckpt in checkpoints:
        size_mb = Path(ckpt).stat().st_size / 1e6
        print(f'    {ckpt} ({size_mb:.1f} MB)')

if logs:
    print(f'  Logs:')
    for log in logs:
        print(f'    {log}')

if summary_path.exists():
    summary = json.load(open(summary_path))
    print(f'\n📈 Pretrained vs. finetuned (PDF Sec 3.1 required comparison):')
    print(f'  Pretrained test exact-match accuracy: {summary["pretrained_test_accuracy"]}')
    print(f'  Finetuned test exact-match accuracy:  {summary["finetuned_test_accuracy"]}')
    print(f'  Best val_loss / val_ppl: {summary["best_val_loss"]:.4f} / {summary["best_val_ppl"]:.2f}')

print(f'\n📝 To resume finetuning in next run:')
print(f'  1. Save this notebook output as a Kaggle dataset')
print(f'  2. Set CHECK_DIR = "/kaggle/input/<this-output-dataset>/finetune_checkpoints"')
print(f'  3. Run the notebook again')



📊 Finetuning outputs:
  Checkpoints:
    /kaggle/working/finetune_checkpoints/checkpoint_best.pt (181.2 MB)
    /kaggle/working/finetune_checkpoints/checkpoint_last.pt (181.2 MB)
  Logs:
    /kaggle/working/finetune_checkpoints/training_bhojpuri.log

📈 Pretrained vs. finetuned (PDF Sec 3.1 required comparison):
  Pretrained test exact-match accuracy: 0.0
  Finetuned test exact-match accuracy:  0.164
  Best val_loss / val_ppl: 8.3672 / 4303.41

📝 To resume finetuning in next run:
  1. Save this notebook output as a Kaggle dataset
  2. Set CHECK_DIR = "/kaggle/input/<this-output-dataset>/finetune_checkpoints"
  3. Run the notebook again


## Final Test-Set Evaluation (Best FINETUNING Checkpoint)

`run_finetuning()` already prints/saves the headline "finetuned test exact-match accuracy" using the best **finetuning** checkpoint (see `finetune_summary.json`). This cell reloads that same checkpoint independently -- `OUT_DIR/checkpoint_best.pt`, i.e. the lowest-val-loss checkpoint saved *during finetuning*, **not** `PRETRAINED_CKPT` (which is a different `checkpoint_best.pt`, from Phase 2 pretraining) -- and breaks the accuracy down by question type and by held-out (`_ho`) vs. seen template phrasing, for the PDF Sec 3.1/3.4 discussion. Note: Bhojpuri templates are best-effort and not yet native-speaker-verified (see `finetune/data/stats.json`), so treat per-category gaps here as directional.

In [7]:
import json
import re
from collections import defaultdict

import torch

from finetune.finetune import evaluate_exact_match
from model.transformer import BhojpuriTransformer
from tokenizer.tokenizer_wrapper import BhojpuriTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BhojpuriTokenizer()
model = BhojpuriTransformer().to(device)

# OUT_DIR/checkpoint_best.pt is the FINETUNING-best checkpoint (lowest val loss seen during
# the finetuning run in the cell above) -- distinct from PRETRAINED_CKPT, which is a
# differently-named-but-identically-filenamed checkpoint_best.pt from Phase 2 pretraining.
best_finetune_ckpt_path = Path(OUT_DIR) / 'checkpoint_best.pt'
assert best_finetune_ckpt_path.exists(), f'{best_finetune_ckpt_path} not found -- run the finetuning cell above first'
ckpt = torch.load(best_finetune_ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"✅ Loaded FINETUNING best checkpoint from {best_finetune_ckpt_path} (step {ckpt['step']}, epoch {ckpt['epoch']})")

# Headline number -- computed the exact same way as finetune_summary.json's finetuned_test_accuracy
test_path = Path(ROOT_DIR) / 'finetune' / 'data' / 'test.jsonl'
overall_acc = evaluate_exact_match(model, tokenizer, str(test_path), device)
print(f"\n📊 Test exact-match accuracy (finetuning-best checkpoint): {overall_acc:.4f}")

# Breakdown: per question-type, and held-out-template vs seen-template generalization gap.
# template_id looks like 'pairwise_value_height_ho' -- strip the attribute + optional held-out
# suffix to get the underlying question pattern (matches stats.json's leakage_avoidance note).
def base_pattern(template_id):
    return re.sub(r'_(age|height|weight|price)(_ho)?$', '', template_id)

examples = [json.loads(l) for l in open(test_path, encoding='utf-8') if l.strip()]
correct_by_group, total_by_group = defaultdict(int), defaultdict(int)
correct_ho = total_ho = correct_seen = total_seen = 0
mistakes = []

with torch.no_grad():
    for ex in examples:
        ids = tokenizer.encode(ex['prompt'], add_special_tokens=False)
        if not ids:
            continue
        input_ids = torch.tensor([ids], device=device)
        generated = model.generate(input_ids, max_new_tokens=10, temperature=1.0, greedy=True)
        gen_ids = generated[0].cpu().numpy().tolist()[len(ids):]
        pred_words = tokenizer.decode(gen_ids).strip().split()
        pred = pred_words[0] if pred_words else ''
        gold = ex['answer'].strip()
        is_correct = pred == gold

        group = base_pattern(ex['template_id'])
        correct_by_group[group] += int(is_correct)
        total_by_group[group] += 1

        if ex['template_id'].endswith('_ho'):
            correct_ho += int(is_correct); total_ho += 1
        else:
            correct_seen += int(is_correct); total_seen += 1

        if not is_correct and len(mistakes) < 10:
            mistakes.append({'prompt': ex['prompt'], 'gold': gold, 'pred': pred})

print(f"\nHeld-out-template phrasing vs seen-template phrasing:")
if total_ho:
    print(f"  Held-out (_ho): {correct_ho}/{total_ho} = {correct_ho/total_ho:.4f}")
if total_seen:
    print(f"  Seen:           {correct_seen}/{total_seen} = {correct_seen/total_seen:.4f}")

print(f"\nBy question type:")
for group in sorted(total_by_group):
    c, t = correct_by_group[group], total_by_group[group]
    print(f"  {group:28s} {c:4d}/{t:4d} = {c/t:.4f}")

print(f"\nSample mistakes (up to 10):")
for m in mistakes:
    print(f"  {m['prompt'][:90]}")
    print(f"    gold={m['gold']!r}  pred={m['pred']!r}")

breakdown = {
    'checkpoint_used': str(best_finetune_ckpt_path),
    'checkpoint_step': ckpt['step'],
    'checkpoint_epoch': ckpt['epoch'],
    'overall_accuracy': overall_acc,
    'held_out_accuracy': correct_ho / total_ho if total_ho else None,
    'seen_template_accuracy': correct_seen / total_seen if total_seen else None,
    'by_question_type': {g: correct_by_group[g] / total_by_group[g] for g in total_by_group},
}
breakdown_path = Path(OUT_DIR) / 'test_eval_breakdown.json'
with open(breakdown_path, 'w', encoding='utf-8') as f:
    json.dump(breakdown, f, indent=2, ensure_ascii=False)
print(f"\n✅ Saved breakdown to {breakdown_path}")

✅ Loaded FINETUNING best checkpoint from /kaggle/working/finetune_checkpoints/checkpoint_best.pt (step 4000, epoch 1)

📊 Test exact-match accuracy (finetuning-best checkpoint): 0.1640

Held-out-template phrasing vs seen-template phrasing:
  Held-out (_ho): 72/504 = 0.1429
  Seen:           256/1496 = 0.1711

By question type:
  equal                          12/ 120 = 0.1000
  pairwise_value                  0/ 600 = 0.0000
  pairwise_yesno                261/ 506 = 0.5158
  three_superlative_max           0/ 197 = 0.0000
  three_superlative_min           0/ 193 = 0.0000
  transitive_max                  0/ 154 = 0.0000
  transitive_min                  0/ 156 = 0.0000
  transitive_yesno               55/  74 = 0.7432

Sample mistakes (up to 10):
  सवाल: चंदा के उमिर 72 बरिस बा। रमेश के उमिर 25 बरिस बा। पार्वती के उमिर 75 बरिस बा। इनमें 
    gold='पार्वती'  pred='मुनियायायायायायायायाया'
  सवाल: बाबू के ऊँचाई 173 सेंटीमीटर बा, उर्मिला के ऊँचाई 141 सेंटीमीटर बा। का बाबू, उर्मिला 
    gol